### dataset normalization
raw flights data already comes denormalized, as one table

this splits the flights csv into the normalized relations from project part 2

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime as dt

#### load data 

In [2]:
airlines = pd.read_csv('airline_dataset/airlines.csv')
airports = pd.read_csv('airline_dataset/airports.csv')

In [3]:
## already normalized
print(airlines.shape)
airlines.head(3)

(14, 2)


,IATA_CODE,AIRLINE
0,UA,United Air Lines Inc.
1,AA,American Airlines Inc.
2,US,US Airways Inc.


In [4]:
## already normalized
print(airports.shape)
airports.head(3)

(322, 7)


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919


In [5]:
## add DOT ID for aiports
airport_dot = pd.read_csv('airline_dataset/T_T100D_MARKET_US_CARRIER_ONLY.csv')
airport_dot = airport_dot[['DEST_AIRPORT_ID','DEST']].drop_duplicates()
airport_dot['DEST_AIRPORT_ID'] = airport_dot['DEST_AIRPORT_ID'].astype(str)

airport_dot.columns=['DOT_ID','IATA_CODE']

print(airport_dot.shape)
airport_dot.head(3)

(1182, 2)


,DOT_ID,IATA_CODE
0,10005,05A
1,10299,ANC
3,11214,CXF


In [6]:
airports = pd.merge(airports, airport_dot, on=['IATA_CODE'], how='left')

print(airports.shape)
airports.head()

(322, 8)


,IATA_CODE,AIRPORT,CITY,STATE,COUNTRY,LATITUDE,LONGITUDE,DOT_ID
0,ABE,Lehigh Valley International Airport,Allentown,PA,USA,40.65236,-75.44040,10135
1,ABI,Abilene Regional Airport,Abilene,TX,USA,32.41132,-99.68190,10136
2,ABQ,Albuquerque International Sunport,Albuquerque,NM,USA,35.04022,-106.60919,10140
3,ABR,Aberdeen Regional Airport,Aberdeen,SD,USA,45.44906,-98.42183,10141
4,ABY,Southwest Georgia Regional Airport,Albany,GA,USA,31.53552,-84.19447,10146


In [7]:
airports.isna().sum()

IATA_CODE    0
AIRPORT      0
CITY         0
STATE        0
COUNTRY      0
LATITUDE     3
LONGITUDE    3
DOT_ID       0
dtype: int64

In [8]:
flights = pd.read_csv('airline_dataset/flights.csv', dtype=str)

print(flights.shape)
flights.head(3)

(5819079, 31)


,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_TIME,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
0,2015,1,1,4,AS,98,N407AS,ANC,SEA,0005,...,0408,-22,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,0010,...,0741,-9,0,0,NaN,NaN,NaN,NaN,NaN,NaN
2,2015,1,1,4,US,840,N171US,SFO,CLT,0020,...,0811,5,0,0,NaN,NaN,NaN,NaN,NaN,NaN


#### create time key from raw date columns

In [9]:
## add leading zero for date strings
flights['MONTH'] = flights['MONTH'].apply(lambda x: '0'+str(x) if len(str(x)) == 1 else str(x))
flights['DAY'] = flights['DAY'].apply(lambda x: '0'+str(x) if len(str(x)) == 1 else str(x))

flights['TIME_KEY'] = flights[['YEAR','MONTH','DAY','DAY_OF_WEEK']].astype(str).sum(axis=1)

In [10]:
flights.head(3)

,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,...,ARRIVAL_DELAY,DIVERTED,CANCELLED,CANCELLATION_REASON,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY,TIME_KEY
0,2015,01,01,4,AS,98,N407AS,ANC,SEA,0005,...,-22,0,0,NaN,NaN,NaN,NaN,NaN,NaN,201501014
1,2015,01,01,4,AA,2336,N3KUAA,LAX,PBI,0010,...,-9,0,0,NaN,NaN,NaN,NaN,NaN,NaN,201501014
2,2015,01,01,4,US,840,N171US,SFO,CLT,0020,...,5,0,0,NaN,NaN,NaN,NaN,NaN,NaN,201501014


In [11]:
flights.info()

<class 'pandas.DataFrame'>
RangeIndex: 5819079 entries, 0 to 5819078
Data columns (total 32 columns):
 #   Column               Dtype
---  ------               -----
 0   YEAR                 str  
 1   MONTH                str  
 2   DAY                  str  
 3   DAY_OF_WEEK          str  
 4   AIRLINE              str  
 5   FLIGHT_NUMBER        str  
 6   TAIL_NUMBER          str  
 7   ORIGIN_AIRPORT       str  
 8   DESTINATION_AIRPORT  str  
 9   SCHEDULED_DEPARTURE  str  
 10  DEPARTURE_TIME       str  
 11  DEPARTURE_DELAY      str  
 12  TAXI_OUT             str  
 13  WHEELS_OFF           str  
 14  SCHEDULED_TIME       str  
 15  ELAPSED_TIME         str  
 16  AIR_TIME             str  
 17  DISTANCE             str  
 18  WHEELS_ON            str  
 19  TAXI_IN              str  
 20  SCHEDULED_ARRIVAL    str  
 21  ARRIVAL_TIME         str  
 22  ARRIVAL_DELAY        str  
 23  DIVERTED             str  
 24  CANCELLED            str  
 25  CANCELLATION_REASON  str  
 2

In [12]:
flights.isna().sum()

YEAR                         0
MONTH                        0
DAY                          0
DAY_OF_WEEK                  0
AIRLINE                      0
FLIGHT_NUMBER                0
TAIL_NUMBER              14721
ORIGIN_AIRPORT               0
DESTINATION_AIRPORT          0
SCHEDULED_DEPARTURE          0
DEPARTURE_TIME           86153
DEPARTURE_DELAY          86153
TAXI_OUT                 89047
WHEELS_OFF               89047
SCHEDULED_TIME               6
ELAPSED_TIME            105071
AIR_TIME                105071
DISTANCE                     0
WHEELS_ON                92513
TAXI_IN                  92513
SCHEDULED_ARRIVAL            0
ARRIVAL_TIME             92513
ARRIVAL_DELAY           105071
DIVERTED                     0
CANCELLED                    0
CANCELLATION_REASON    5729195
AIR_SYSTEM_DELAY       4755640
SECURITY_DELAY         4755640
AIRLINE_DELAY          4755640
LATE_AIRCRAFT_DELAY    4755640
WEATHER_DELAY          4755640
TIME_KEY                     0
dtype: i

In [13]:
## flights with null aircraft id were cancelled 
flights[flights['TAIL_NUMBER'].isna()]['CANCELLATION_REASON'].value_counts(dropna=False)

## keep these records in, and fill in TAIL_NUMBER to avoid nulls in primary key 
flights['TAIL_NUMBER'].fillna('NA', inplace=True)

/var/folders/d0/q4rwdhyx3bg4m8jmrnyj715r0000gn/T/ipykernel_93738/3639672897.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  flights['TAIL_NUMBER'].fillna('NA', inplace=True)


0          N407AS
1          N3KUAA
2          N171US
3          N3HYAA
4          N527AS
            ...  
5819074    N657JB
5819075    N828JB
5819076    N913JB
5819077    N527JB
5819078    N534JB
Name: TAIL_NUMBER, Length: 5819079, dtype: str

In [14]:
flights['TAIL_NUMBER'].dropna().apply(lambda x: len(x)).value_counts()

TAIL_NUMBER
6    5789255
5      15103
Name: count, dtype: int64

In [15]:
## primary keys
primary_keys = ['TIME_KEY','AIRLINE','FLIGHT_NUMBER','TAIL_NUMBER','ORIGIN_AIRPORT','DESTINATION_AIRPORT']

#### create time key relation


In [16]:
time_key_df = flights[['TIME_KEY','YEAR','MONTH','DAY','DAY_OF_WEEK']].drop_duplicates().reset_index(drop=True)
time_key_df.head(3)

,TIME_KEY,YEAR,MONTH,DAY,DAY_OF_WEEK
0,201501014,2015,01,01,4
1,201501025,2015,01,02,5
2,201501036,2015,01,03,6


#### create flight_status relation

In [17]:
flight_status_df = flights[primary_keys + ['DIVERTED','CANCELLED','CANCELLATION_REASON']].drop_duplicates().reset_index(drop=True)

print(flight_status_df.shape)
flight_status_df.head()

(5819079, 9)


,TIME_KEY,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DIVERTED,CANCELLED,CANCELLATION_REASON
0,201501014,AS,98,N407AS,ANC,SEA,0,0,NaN
1,201501014,AA,2336,N3KUAA,LAX,PBI,0,0,NaN
2,201501014,US,840,N171US,SFO,CLT,0,0,NaN
3,201501014,AA,258,N3HYAA,LAX,MIA,0,0,NaN
4,201501014,AS,135,N527AS,SEA,ANC,0,0,NaN


#### create flight_delays relation

In [18]:
flight_delays_df = flights[primary_keys + ['AIR_SYSTEM_DELAY','SECURITY_DELAY','AIRLINE_DELAY','LATE_AIRCRAFT_DELAY','WEATHER_DELAY']].drop_duplicates().reset_index(drop=True)
flight_delays_df.dropna(inplace=True)

print(flight_delays_df.shape)
flight_delays_df.head(3)

(1063439, 11)


,TIME_KEY,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,AIR_SYSTEM_DELAY,SECURITY_DELAY,AIRLINE_DELAY,LATE_AIRCRAFT_DELAY,WEATHER_DELAY
27,201501014,NK,597,N528NK,MSP,FLL,25,0,0,0,0
30,201501014,NK,168,N629NK,PHX,ORD,43,0,0,0,0
35,201501014,HA,17,N389HA,LAS,HNL,0,0,15,0,0


#### create flights relation

In [19]:
flights_df = flights[primary_keys + ['SCHEDULED_DEPARTURE','DEPARTURE_TIME','DEPARTURE_DELAY',
                                     'TAXI_OUT','WHEELS_OFF','SCHEDULED_TIME','ELAPSED_TIME','AIR_TIME',
                                     'DISTANCE','WHEELS_ON','TAXI_IN','SCHEDULED_ARRIVAL','ARRIVAL_TIME','ARRIVAL_DELAY']].drop_duplicates().reset_index(drop=True)

print(flights_df.shape)
flights_df.head(3)

(5819079, 20)


,TIME_KEY,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
0,201501014,AS,98,N407AS,ANC,SEA,0005,2354,-11,21,0015,205,194,169,1448,0404,4,0430,0408,-22
1,201501014,AA,2336,N3KUAA,LAX,PBI,0010,0002,-8,12,0014,280,279,263,2330,0737,4,0750,0741,-9
2,201501014,US,840,N171US,SFO,CLT,0020,0018,-2,16,0034,286,293,266,2296,0800,11,0806,0811,5


In [20]:
### clean flight times

flight_time_cols = ['SCHEDULED_DEPARTURE','DEPARTURE_TIME','WHEELS_OFF','WHEELS_ON','SCHEDULED_ARRIVAL','ARRIVAL_TIME']

for ftc in flight_time_cols:
    print(ftc)
    flights_df[ftc] = pd.to_datetime(flights_df[ftc].astype(str).str.zfill(4), format="%H%M", errors='coerce').dt.time

SCHEDULED_DEPARTURE
DEPARTURE_TIME
WHEELS_OFF
WHEELS_ON
SCHEDULED_ARRIVAL
ARRIVAL_TIME


In [21]:
flights_df.head(3)

,TIME_KEY,AIRLINE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_TIME,DEPARTURE_DELAY,TAXI_OUT,WHEELS_OFF,SCHEDULED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,WHEELS_ON,TAXI_IN,SCHEDULED_ARRIVAL,ARRIVAL_TIME,ARRIVAL_DELAY
0,201501014,AS,98,N407AS,ANC,SEA,00:05:00,23:54:00,-11,21,00:15:00,205,194,169,1448,04:04:00,4,04:30:00,04:08:00,-22
1,201501014,AA,2336,N3KUAA,LAX,PBI,00:10:00,00:02:00,-8,12,00:14:00,280,279,263,2330,07:37:00,4,07:50:00,07:41:00,-9
2,201501014,US,840,N171US,SFO,CLT,00:20:00,00:18:00,-2,16,00:34:00,286,293,266,2296,08:00:00,11,08:06:00,08:11:00,5


#### create cancelled_reason mapping relation


In [22]:
## from the data definitions in kaggle
cancellation_df = pd.DataFrame([['A', 'Airline/Carrier'],['B','Weather'],['C','National Air System'],['D','Security']],
                               columns=['CANCELLATION_REASON','VALUE'])
cancellation_df

,CANCELLATION_REASON,VALUE
0,A,Airline/Carrier
1,B,Weather
2,C,National Air System
3,D,Security


### export dataframes

In [23]:
## already normalized
airlines.to_csv('airline_dataset_normalized/airlines.csv', index=None)

## added DOT foreign key
airports.to_csv('airline_dataset_normalized/airports.csv', index=None)

## newly normalized
time_key_df.to_csv('airline_dataset_normalized/time_key.csv', index=None)
flight_status_df.to_csv('airline_dataset_normalized/flight_status.csv', index=None)
flight_delays_df.to_csv('airline_dataset_normalized/flight_delays.csv', index=None)
flights_df.to_csv('airline_dataset_normalized/flights.csv', index=None)
cancellation_df.to_csv('airline_dataset_normalized/cancellation_reason.csv', index=None)

### SQL SCHEMA EXAMPLES
if there was a live database, could read files from datalake into datawarehouse

below code (with a valid connection), would create tables in the datawarehouse, and then could load new files as they come in

In [ ]:
import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from sqlalchemy.ext.declarative import declarative_base

Base = declarative_base()

In [ ]:

class Airlines(Base):

    __tablename__ = 'dim_airlines'

    iata_code = sa.Column('iata_code', sa.VARCHAR(50), primary_key=True)
    airline_name = sa.Column('airline_name',sa.VARCHAR(255))


class Airports(Base):

    __tablename__ = 'dim_airports'

    iata_code = sa.Column('iata_code', sa.VARCHAR(50), primary_key=True)
    airport = sa.Column('airport',sa.VARCHAR(255))
    city = sa.Column('city', sa.VARCHAR(255))
    state = sa.Column('state', sa.VARCHAR(255))
    country = sa.Column('country', sa.VARCHAR(255))
    latitude = sa.Column('latitude', sa.FLOAT)
    longitude = sa.Column('longitude', sa.FLOAT)
    dot_id = sa.Column('dot_id', sa.VARCHAR(255))


class TimeKey(Base):

    __tablename__ = 'dim_time_key'

    time_key = sa.Column('time_key',sa.VARCHAR(50), primary_key=True)
    year = sa.Column('year', sa.VARCHAR(10))
    month = sa.Column('month', sa.VARCHAR(10))
    day = sa.Column('day', sa.VARCHAR(10))
    day_of_week = sa.Column('day_of_week', sa.INT)
    sa.Index('idx_time_key', time_key)


class FlightStaus(Base):

    __tablename__ = 'dim_flight_status'

    time_key = sa.Column('time_key', sa.VARCHAR(50), primary_key=True)
    airline = sa.Column('airline',sa.VARCHAR(255), primary_key=True)
    flight_number = sa.Column('flight_number', sa.VARCHAR(50), primary_key=True)
    tail_number = sa.Column('tail_number', sa.VARCHAR(50), primary_key=True)
    origin_airport = sa.Column('origin_airport', sa.VARCHAR(255), primary_key=True)
    destination_airport = sa.Column('destination_airport', sa.VARCHAR(255), primary_key=True)
    diverted = sa.Column('diverted', sa.BOOLEAN)
    cancelled = sa.Column('cancelled', sa.BOOLEAN)
    cancellation_reason = sa.Column('cancellation_reason', sa.VARCHAR(10))
    sa.Index('idx_time_airline', time_key, airline)


class FlightDelays(Base):

    __tablename__ = 'fct_flight_delays'

    time_key = sa.Column('time_key', sa.VARCHAR(50), primary_key=True)
    airline = sa.Column('airline',sa.VARCHAR(255), primary_key=True)
    flight_number = sa.Column('flight_number', sa.VARCHAR(50), primary_key=True)
    tail_number = sa.Column('tail_number', sa.VARCHAR(50), primary_key=True)
    origin_airport = sa.Column('origin_airport', sa.VARCHAR(255), primary_key=True)
    destination_airport = sa.Column('destination_airport', sa.VARCHAR(255), primary_key=True)
    air_system_delay = sa.Column('air_system_delay',sa.FLOAT)
    security_delay = sa.Column('security_delay', sa.FLOAT)
    airline_delay = sa.Column('airline_delay', sa.FLOAT)
    late_aircraft_delay = sa.Column('late_aircraft_delay', sa.FLOAT)
    weather_delay = sa.Column('weather_delay', sa.FLOAT)
    sa.Index('idx_time_airline', time_key, airline)


class Flights(Base):

    __tablename__ = 'fct_flights'

    time_key = sa.Column('time_key', sa.VARCHAR(50), primary_key=True)
    airline = sa.Column('airline',sa.VARCHAR(255), primary_key=True)
    flight_number = sa.Column('flight_number', sa.VARCHAR(50), primary_key=True)
    tail_number = sa.Column('tail_number', sa.VARCHAR(50), primary_key=True)
    origin_airport = sa.Column('origin_airport', sa.VARCHAR(255), primary_key=True)
    destination_airport = sa.Column('destination_airport', sa.VARCHAR(255), primary_key=True)
    scheduled_departure = sa.Column('scheduled_departure', sa.TIME)
    departure_time = sa.Column('departure_time', sa.TIME)
    departure_delay = sa.Column('departure_delay', sa.INT)
    taxi_out = sa.Column('taxi_out', sa.INT)
    wheels_off = sa.Column('wheels_off', sa.TIME)
    scheduled_time = sa.Column('scheduled_time', sa.INT)
    elapsed_time = sa.Column('elapsed_time', sa.INT)
    air_time = sa.Column('air_time',sa.INT)
    distance = sa.Column('distance', sa.INT)
    wheels_on = sa.Column('wheels_on',  sa.TIME)
    taxi_in = sa.Column('taxi_in', sa.INT)
    scheduled_arrival = sa.Column('scheduled_arrival', sa.TIME)
    arrival_time = sa.Column('arrival_time', sa.TIME)
    arrival_delay = sa.Column('arrival_delay', sa.INT)
    sa.Index('idx_time_airline', time_key, airline)


class CancellationReason(Base):

    __tablename__ = 'dim_cancellation_reason'

    cancellation_reason = sa.Column('cancellation_reason',sa.VARCHAR(10), primary_key=True)
    value = sa.Column('value',sa.VARCHAR(255))



# Base.metadata.create_all()